In [ ]:
import sys
sys.executable

In [ ]:
import os
os.getcwd()

In [ ]:
import torch
import torchvision

In [ ]:
import re
import io
import json
import numpy as np
import pandas as pd
from PIL import Image
import webdataset as wds

In [ ]:
sys.path.append(os.path.join(os.path.abspath(''), '..', '..', '..'))
import config
sys.path.append(config.COCO_UTILS_DIR)
from pycocotools.coco import COCO
from coco_utils import ConvertCocoPolysToMask

In [ ]:
# 路径配置
ROOT = config.CONTEXT_REASONING_RES_DIR
COCO_IMAGE_DIR = config.COCO_TRAIN_IMAGES
BBOX_IMAGE_DIR = config.TRAINING_IMAGES_WITH_BBOX
OUTPUT_PATH = 'size_webdataset/train-%06d.tar'
SHARD_SIZE = 1000

os.makedirs('size_webdataset', exist_ok=True)

# CSV文件路径
CSV_PATHS = {
    'internvl': ROOT + 'internvl/context_reasoning_internvl.csv',
    'molmo': ROOT + 'molmo/context_reasoning_molmo.csv',
    'qwen': ROOT + 'qwen/context_reasoning_qwen.csv'
}

In [ ]:
print("Loading COCO datasets...")
coco = COCO(config.COCO_TRAIN_ANNOTATIONS)
coco_data = torchvision.datasets.CocoDetection(
    config.COCO_TRAIN_IMAGES,
    config.COCO_TRAIN_ANNOTATIONS
)
print(f"✅ COCO dataset loaded: {len(coco_data)} images")

In [ ]:
location_pattern = r'\*?\*?Location:?\*?\*?\s*(.*?)(?=\n(?:\*?\*?Co-occurrence|\*?\*?Size|\*?\*?Summary|$))'
size_pattern = r'\*?\*?Size:?\*?\*?\s*(.*?)(?=\n(?:\*?\*?Co-occurrence|\*?\*?Location|\*?\*?Summary|$))'
co_occurrence_pattern = r'\*?\*?Co-occurrence:?\*?\*?\s*(.*?)(?=\n(?:\*?\*?Location|\*?\*?Size|\*?\*?Summary|$))'

In [ ]:
def parse_response(response):
    """解析response，提取Location, Co-occurrence, Size和Final decision"""
    result = {
        'Location': None,
        'Co-occurrence': None,
        'Size': None,
        'Final_decision': None,
    }
    
    location_match = re.search(r'Location:\s*(In-context|Out-of-context)', response, re.IGNORECASE)
    cooccurrence_match = re.search(r'Co-occurrence:\s*(In-context|Out-of-context)', response, re.IGNORECASE)
    size_match = re.search(r'Size:\s*(In-context|Out-of-context)', response, re.IGNORECASE)
    final_match = re.search(r'Final decision:\s*(In-context|Out-of-context)', response, re.IGNORECASE)
    
    if location_match:
        result['Location'] = location_match.group(1)
    if cooccurrence_match:
        result['Co-occurrence'] = cooccurrence_match.group(1)
    if size_match:
        result['Size'] = size_match.group(1)
    if final_match:
        result['Final_decision'] = final_match.group(1)
    
    return result

In [ ]:
def extract_size_reasoning(response):
    """从response中提取Size相关的reasoning文本"""
    size_pattern = r'Size: (.*?)(?=\n(?:Co-occurrence|Location|Summary|$))'
    size_match = re.search(size_pattern, response, re.DOTALL)
    if size_match:
        return size_match.group(1).strip()
    return ""

In [ ]:
def process_dataframe(df, model_name):
    """处理DataFrame，添加解析后的列"""
    parsed_data = df['response'].apply(parse_response)
    parsed_df = pd.DataFrame(parsed_data.tolist())
    df_processed = pd.concat([df, parsed_df], axis=1)
    df_processed['model'] = model_name
    return df_processed

In [ ]:
print("\nLoading and processing CSV files...")
df_internvl = pd.read_csv(CSV_PATHS['internvl']).drop(columns=['Unnamed: 0'], errors='ignore')
df_molmo = pd.read_csv(CSV_PATHS['molmo']).drop(columns=['Unnamed: 0'], errors='ignore')
df_qwen = pd.read_csv(CSV_PATHS['qwen']).drop(columns=['Unnamed: 0'], errors='ignore')

df_internvl_processed = process_dataframe(df_internvl, 'InternVL')
df_molmo_processed = process_dataframe(df_molmo, 'Molmo')
df_qwen_processed = process_dataframe(df_qwen, 'Qwen')

In [ ]:
print("\nMerging dataframes...")
df_merged = df_internvl_processed[['coco_index', 'Location', 'Co-occurrence', 'Size', 'Final_decision']].merge(
    df_molmo_processed[['coco_index', 'Location', 'Co-occurrence', 'Size', 'Final_decision']], 
    on='coco_index', 
    suffixes=('_internvl', '_molmo')
).merge(
    df_qwen_processed[['coco_index', 'Location', 'Co-occurrence', 'Size', 'Final_decision']], 
    on='coco_index'
)

df_merged.columns = ['coco_index', 
                     'Location_internvl', 'Co-occurrence_internvl', 'Size_internvl', 'Final_decision_internvl',
                     'Location_molmo', 'Co-occurrence_molmo', 'Size_molmo', 'Final_decision_molmo',
                     'Location_qwen', 'Co-occurrence_qwen', 'Size_qwen', 'Final_decision_qwen']

In [ ]:
df_ooc_size = df_merged[
    # (df_merged['Size_internvl'] == 'Out-of-context') &
    # (df_merged['Size_molmo'] == 'Out-of-context') &
    (df_merged['Size_qwen'] == 'Out-of-context')
]

# df_ooc_final = df_merged[
#     # (df_merged['Final_decision_internvl'] == 'Out-of-context') &
#     # (df_merged['Final_decision_molmo'] == 'Out-of-context') &
#     (df_merged['Final_decision_qwen'] == 'Out-of-context')
# ]

# out_of_context_samples = pd.merge(df_ooc_size, df_ooc_final, how='inner')
out_of_context_samples = df_ooc_size
print(f"✅ Out-of-context samples: {len(out_of_context_samples)}")

In [ ]:
df_ic_size = df_merged[
    # (df_merged['Size_internvl'] == 'In-context') &
    # (df_merged['Size_molmo'] == 'In-context') &
    (df_merged['Size_qwen'] == 'In-context')
]

# df_ic_final = df_merged[
#     # (df_merged['Final_decision_internvl'] == 'In-context') &
#     # (df_merged['Final_decision_molmo'] == 'In-context') &
#     (df_merged['Final_decision_qwen'] == 'In-context')
# ]

# in_context_samples = pd.merge(df_ic_size, df_ic_final, how='inner')
in_context_samples = df_ic_size

print(f"✅ In-context samples: {len(in_context_samples)}")

In [ ]:
min_samples = min(len(out_of_context_samples), len(in_context_samples))
print(f"\n⚖️  Balancing dataset: using {min_samples} samples per class")

# 随机采样（可选：设置random_state保证可复现）
out_of_context_samples = out_of_context_samples.sample(n=min_samples, random_state=42)
in_context_samples = in_context_samples.sample(n=min_samples, random_state=42)

# 合并并添加标签列
out_of_context_samples['label'] = out_of_context_samples['Size_qwen']
in_context_samples['label'] = in_context_samples['Size_qwen']

all_samples = pd.concat([out_of_context_samples, in_context_samples], ignore_index=True)

# 打乱顺序
all_samples = all_samples.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"✅ Total balanced samples: {len(all_samples)}")
print(f"   - Out-of-context: {len(out_of_context_samples)}")
print(f"   - In-context: {len(in_context_samples)}")

In [ ]:
all_samples

In [ ]:
print(f"\nCreating WebDataset at {OUTPUT_PATH}...")

In [ ]:
with wds.ShardWriter(OUTPUT_PATH, maxcount=SHARD_SIZE) as sink:
    for idx, row in all_samples.iterrows():
        coco_idx = row['coco_index']
        label = row['label']
        
        # 1. 读取原始COCO图片
        coco_img_id = coco.getImgIds()[coco_idx]
        coco_img_info = coco.loadImgs(coco_img_id)[0]
        original_img_path = os.path.join(COCO_IMAGE_DIR, coco_img_info['file_name'])
        
        with open(original_img_path, 'rb') as f:
            original_img_bytes = f.read()
        
        # 2. 读取bbox图片
        bbox_img_path = f"{BBOX_IMAGE_DIR}/{coco_idx}.png"
        with open(bbox_img_path, 'rb') as f:
            bbox_img_bytes = f.read()
        
        # 3. 获取三个模型的完整response
        internvl_response = df_internvl[df_internvl['coco_index'] == coco_idx]['response'].values[0]
        molmo_response = df_molmo[df_molmo['coco_index'] == coco_idx]['response'].values[0]
        qwen_response = df_qwen[df_qwen['coco_index'] == coco_idx]['response'].values[0]
        
        qwen_size_match = re.search(size_pattern, qwen_response, re.DOTALL)

        # internvl_location_text = internvl_location_match.group(1).strip()
        # molmo_location_text = molmo_location_match.group(1).strip()
        qwen_size_text = qwen_size_match.group(1).strip()

        
        # 4. 构建标注数据
        annotations = {
            'coco_index': int(coco_idx),
            'coco_image_id': int(coco_img_id),
            'file_name': coco_img_info['file_name'],
            'label': label,  # 'In-context' 或 'Out-of-context'
            'qwen_response': qwen_size_text
        }
        
        # 5. 写入WebDataset（去掉 cls 字段）
        sample = {
            "__key__": f"{coco_idx:06d}",
            "original.jpg": original_img_bytes,
            "bbox.png": bbox_img_bytes,
            "json": annotations  # label 已经在这里了
        }
        sink.write(sample)
        
        if (idx + 1) % 100 == 0:
            print(f"  Processed {idx + 1}/{len(all_samples)} samples...")

In [ ]:
print(f"\n✅ WebDataset created successfully!")
print(f"   Total samples: {len(all_samples)}")
print(f"   Output: {OUTPUT_PATH}")

In [ ]:
# ============================================================================
# 7. 验证WebDataset
# ============================================================================
print("\n" + "="*80)
print("验证WebDataset...")
print("="*80)

dataset = wds.WebDataset("size_webdataset/train-000000.tar").decode()

label_counts = {'In-context': 0, 'Out-of-context': 0}

for i, sample in enumerate(dataset):
    label = sample['json']['label']
    label_counts[label] += 1
    
    if i < 2:  # 显示前2个样本
        print(f"\n样本 {i}:")
        print(f"  Key: {sample['__key__']}")
        print(f"  Label: {label}")
        print(f"  COCO Index: {sample['json']['coco_index']}")
        print(f"  File name: {sample['json']['file_name']}")
        print(f"  qwen response preview: {sample['json']['qwen_response']}...")

print(f"\n标签分布:")
print(f"  In-context: {label_counts['In-context']}")
print(f"  Out-of-context: {label_counts['Out-of-context']}")
print("\n✅ 验证完成!")

In [ ]:
sample['json']['qwen_response']